### Data Setup

#### A1. Load data

In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
abalone = fetch_ucirepo(id=1) 
  
# data (as pandas dataframes) 
X = abalone.data.features 
y = abalone.data.targets 

In [ ]:
print(f'Number of rows: {X.shape[0]}')
print(f'Number of columns: {X.shape[1] + 1}') # +1 for target column

Number of rows: 4177
Number of columns: 9


In [18]:
print(X.columns)
print(y.columns)

Index(['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight',
       'Viscera_weight', 'Shell_weight'],
      dtype='str')
Index(['Rings'], dtype='str')


Checkpoint 1
- What is input? 
    Sex, Length, diameter, height, whole weight, shucked weight, viscera weight, shell weight

- What is output?
    Rings

- why output is numeric?
    Because we are performing the regression and predicting age.


#### A2. Convert target

In [20]:
print(f"Initial y: {y}")
y = y + 1.5
print(f"Converted y: {y}")

Initial y:       Rings
0      16.5
1       8.5
2      10.5
3      11.5
4       8.5
...     ...
4172   12.5
4173   11.5
4174   10.5
4175   11.5
4176   13.5

[4177 rows x 1 columns]
Converted y:       Rings
0      18.0
1      10.0
2      12.0
3      13.0
4      10.0
...     ...
4172   14.0
4173   13.0
4174   12.0
4175   13.0
4176   15.0

[4177 rows x 1 columns]


#### A3. Choose features

In [ ]:
# Justification:
# Feature 1: Shell weight (Strongest indicator of age as layers accumulate over time)
# Feature 2: Length (Primary growth measurement in horizontal dir)
# Feature 3: Diameter (Perpendicular measurement to length to capture overall size) 

X = X[['Shell_weight', 'Length', 'Diameter']] 
X.head()

,Shell_weight,Length,Diameter
0,0.150,0.455,0.365
1,0.070,0.350,0.265
2,0.210,0.530,0.420
3,0.155,0.440,0.365
4,0.055,0.330,0.255


#### A4. Train-test split

In [44]:
import numpy as np

np.random.seed(42)
indices = np.random.permutation(len(X))

train_size = int(0.8 * len(X))

train_idx = indices[:train_size]
test_idx = indices[train_size:]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (3341, 3), y_train shape: (3341, 1)
X_test shape: (836, 3), y_test shape: (836, 1)


#### A5. Normalize inputs

In [26]:
X_mean = X_train.mean()
X_std = X_train.std()

X_train_norm = (X_train - X_mean) / X_std
X_test_norm = (X_test - X_mean) / X_std

Checkpoint 2:
why normalization is needed for learning?
- It helps in scaling all the features in the same range.
- Prevents feat with large value to dominating the weights
- gradient converge converges faster

### Part B. Define Model

In [28]:
def forward(X, w, b):
    y_hat = np.dot(X, w) + b
    return y_hat

In [40]:
d = X_train_norm.shape[1] # number of feat
w = np.zeros((d, 1))
b = 0.0

y_hat = forward(X_train_norm, w, b)

print(f"Shape of X: {X_train_norm.shape}")
print(f"Shape of w: {w.shape}")
print(f"Shape oof b: {np.array(b).shape})")
print(f"Shape of y_hat: {y_hat.shape}")

Shape of X: (3341, 3)
Shape of w: (3, 1)
Shape oof b: ())
Shape of y_hat: (3341, 1)


Checkpoint 3:

Number of learnable param = 4 (3 weights for 3 features and 1 bias)

### Part C: Define loss (MSE)

In [35]:
def mse(y, y_hat):
    loss = np.mean((y - y_hat) ** 2)
    return loss

Checkpoint 4:

Why square: It makes all error positive and helps in converging

what mistakes are expensive: large mistakes are expensive as error of 10 is penalized 100 times

### Part D: Learning rule

Checkpoint 5:

what gradient means in words: Gradient is the multidimensional slope of the loss function; it points in the dir of the steepest increase of the error

why subtracting gradient reduces loss: Since gradient points uphill toward max error, subtracting it forces our params to move in the exact opp. dir (downhill) toward the min error

In [37]:
# implement gradient
def grad_w(X, y, y_hat):
    n = len(y)
    err = y_hat-y
    dw = (2/n) * np.dot(X.T, err)
    
    return dw

def grad_b(y, y_hat):
    n = len(y)
    err = y_hat-y
    db = (2/n) * np.sum(err)
    
    return db

Checkpoint 6:

meaning of large gradient: large gradient means the error is currently very steep. It indicates the model is making significant errors and needs a large correction to get closer to the minima

effect of too-large learning rate: If the learning rate is too large, the update step will be massive. The model might overshoot the minimum completely, causing the loss to oscillate wildly or even explode to infinity (divergence)

### Part E: Training loop

In [42]:
np.random.seed(42)

# initialize w (small random values)
d = X_train_norm.shape[1]
w = np.random.randn(d, 1) * 0.01

# initialize b (zero)
b = 0.0

# choose learning rate and epochs
learning_rate = 0.1
epochs = 100
k = 10 # print every 10 epochs

for epoch in range(epochs):
    # 1) forward pass
    y_hat = forward(X_train_norm, w, b)
    
    # 2) compute loss
    loss = mse(y_train, y_hat)
    
    # 3) compute gradients
    dw = grad_w(X_train_norm, y_train, y_hat)
    db = grad_b(y_train, y_hat)
    
    # 4) update w and b
    w = w - (learning_rate * dw)
    b = b - (learning_rate * db)
    
    if epoch % k == 0: # [cite: 155]
        print(f"Epoch {epoch}: Loss = {loss:.4f}")
        
# Final loss print
print(f"Final Training Loss after {epochs} epochs: {loss:.4f}")

Epoch 0: Loss = 178.0380
Epoch 10: Loss = 8.6057
Epoch 20: Loss = 6.6040
Epoch 30: Loss = 6.5242
Epoch 40: Loss = 6.4887
Epoch 50: Loss = 6.4668
Epoch 60: Loss = 6.4529
Epoch 70: Loss = 6.4437
Epoch 80: Loss = 6.4373
Epoch 90: Loss = 6.4327
Final Training Loss after 100 epochs: 6.4294


Checkpoint 7:

Initial expectation: I expect the loss to drop very fast in the first 10-20 epochs and then slow down as it approaches the minimum.

Revised expectation after training: It happens the same as expected initially.

### Part F: Evaluation

In [46]:
# prediction for test
y_test_hat = forward(X_test_norm, w, b)

# compute test MSE
test_mse = mse(y_test, y_test_hat)

# compute test MAE
test_mae = np.mean(np.abs(y_test - y_test_hat))

print(f"Test MSE: {test_mse:.4f}")
print(f"Test MAE: {test_mae:.4f}")

Test MSE: 5.6497
Test MAE: 1.7733


In [49]:
# print 5 example predictions
y_test_np = y_test.to_numpy().reshape(-1, 1)

print("5 Example Predictions:")
for i in range(5):
    true_age = y_test_np[i][0] 
    pred_age = y_test_hat[i][0]
    abs_error = abs(true_age - pred_age)
    print(f"Example {i+1} | True Age: {true_age:.1f} | Predicted Age: {pred_age:.1f} | Absolute Error: {abs_error:.1f}")

5 Example Predictions:
Example 1 | True Age: 12.0 | Predicted Age: 11.1 | Absolute Error: 0.9
Example 2 | True Age: 13.0 | Predicted Age: 14.7 | Absolute Error: 1.7
Example 3 | True Age: 12.0 | Predicted Age: 13.3 | Absolute Error: 1.3
Example 4 | True Age: 13.0 | Predicted Age: 10.7 | Absolute Error: 2.3
Example 5 | True Age: 9.0 | Predicted Age: 10.2 | Absolute Error: 1.2


Checkpoint 8:

Systematic error: Model guesses too high for young abalones and too low for old ones because it can only draw a straight line

Observed bias: Model mostly just guesses numbers that are close to the average age